# Объединённое исследование target и baseline-модели

**Цель:** В одном ноутбуке выполнить полный цикл: генерация target-кандидатов, leakage-check, сравнение baseline-моделей и выбор финальной target/модели.

**Target-кандидаты:** **tb_vol_5bar**, **tp_sl 2%/1%**, **tp_sl 1%/0.5%** (nikitapre).

**Backtest:** HOLD-зона (proba 0.4–0.6), комиссия только при смене позиции.

**Фичи:** volatility, returns, momentum, RSI, MACD, Bollinger, **rd_mom** (RD), volume, OHLC, time.

---

**Финал:** выбранный target = **tp_sl_1_05**, базовая модель = **LightGBM**; сохранение артефактов — в последнем разделе.

## 1. Импорты и загрузка

In [1]:
import sys
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
from sklearn.dummy import DummyClassifier
import warnings
warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments','fork') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

from src.features.feature_pipeline import add_features, get_feature_columns, FEATURE_COLS

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
try:
    import catboost as cb
    HAS_CB = True
except ImportError:
    HAS_CB = False

print(f'LightGBM: {HAS_LGB}, CatBoost: {HAS_CB}')

LightGBM: True, CatBoost: True


In [2]:
prepared_path = os.path.join(_root, 'outputs', 'prepared_with_rd_regime.parquet')
if not os.path.exists(prepared_path):
    raise FileNotFoundError(f'Запустите 01_data_prep/02_Load_And_Prepare_Data.ipynb. Файл не найден: {prepared_path}')

df = pd.read_parquet(prepared_path)
# Фичи могут отсутствовать в старом parquet — добавляем при необходимости
need_cols = ['volatility_14', 'rsi_14', 'macd_hist', 'rd_mom_1']
if any(c not in df.columns for c in need_cols):
    df, _ = add_features(df)
print(f'Загружено: {len(df):,} строк, {df["session_key"].nunique()} сессий')
print(f'Колонок: {len(df.columns)}')

Загружено: 365,375 строк, 1048 сессий
Колонок: 40


## 2. Target: tb_vol_5bar, tp_sl 2%/1%, tp_sl 1%/0.5%

In [3]:
def add_tb_vol_5bar(df, by='session_key', vol_col='volatility_14', mult=1.5, h=5):
    """Triple-barrier: TP/SL = mult*vol, horizon=h."""
    df = df.copy()
    labels = pd.Series(index=df.index, dtype=np.float64)
    for skey, g in df.groupby(by, sort=False):
        idx = g.index
        c, h_arr, l_arr = g['close_price'].values, g['high'].values, g['low'].values
        v = g[vol_col].fillna(0.01).clip(lower=1e-6).values
        lab = np.full(len(g), np.nan, dtype=np.float64)
        for i in range(len(g) - h):
            up, dn = c[i] * (1 + mult * v[i]), c[i] * (1 - mult * v[i])
            for j in range(1, h + 1):
                if h_arr[i + j] >= up: lab[i] = 1; break
                if l_arr[i + j] <= dn: lab[i] = -1; break
            else: lab[i] = 0
        labels.loc[idx] = lab
    df['tb_vol_5bar'] = labels.values
    return df

def add_target_tp_sl_fixed(df, by='session_key', TP_pct=0.02, SL_pct=0.01, H=20, out_col='tp_sl_fixed'):
    """TP/SL фикс. %: ambiguous intrabar (TP и SL в одном баре) -> 0."""
    df = df.copy()
    labels = pd.Series(index=df.index, dtype=np.float64)
    for skey, g in df.groupby(by, sort=False):
        idx, c = g.index, g['close_price'].values
        h_arr, l_arr = g['high'].values, g['low'].values
        lab = np.full(len(g), np.nan, dtype=np.float64)
        for i in range(len(g) - H):
            entry = c[i]
            tp_up, sl_down = entry * (1 + TP_pct), entry * (1 - SL_pct)
            tp_dn, sl_up = entry * (1 - TP_pct), entry * (1 + SL_pct)

            res_long = 0
            for j in range(1, H + 1):
                hi, lo = h_arr[i + j], l_arr[i + j]
                hit_tp_long = hi >= tp_up
                hit_sl_long = lo <= sl_down
                if hit_tp_long and hit_sl_long:
                    res_long = 0
                    break
                if hit_tp_long:
                    res_long = 1
                    break
                if hit_sl_long:
                    res_long = -1
                    break

            res_short = 0
            for j in range(1, H + 1):
                hi, lo = h_arr[i + j], l_arr[i + j]
                hit_tp_short = lo <= tp_dn
                hit_sl_short = hi >= sl_up
                if hit_tp_short and hit_sl_short:
                    res_short = 0
                    break
                if hit_tp_short:
                    res_short = 1
                    break
                if hit_sl_short:
                    res_short = -1
                    break

            if res_long == 1 and res_short != 1:
                lab[i] = 1
            elif res_short == 1 and res_long != 1:
                lab[i] = -1
            else:
                lab[i] = 0
        labels.loc[idx] = lab
    df[out_col] = labels.values
    return df

df = add_tb_vol_5bar(df)
df = add_target_tp_sl_fixed(df, TP_pct=0.02, SL_pct=0.01, H=20, out_col='tp_sl_2_1')
df = add_target_tp_sl_fixed(df, TP_pct=0.01, SL_pct=0.005, H=20, out_col='tp_sl_1_05')
TARGETS = ['tb_vol_5bar', 'tp_sl_2_1', 'tp_sl_1_05']
for tcol in TARGETS:
    v = df.dropna(subset=[tcol]).copy()
    v = v[v[tcol].isin([-1.0, 1.0])]
    print(f'{tcol}: BUY={(v[tcol]==1).sum():,} SELL={(v[tcol]==-1).sum():,}')

tb_vol_5bar: BUY=139,094 SELL=137,635
tp_sl_2_1: BUY=54,383 SELL=47,360


tp_sl_1_05: BUY=97,874 SELL=95,494


## 3. Leakage check

Проверяем, что target не является прямой функцией RD-фичей (`rd_regime`, `rd_mom_1`, `ret_1`).
Корреляции не должны быть близки к 1 по модулю.

In [4]:
cols_check = ['rd_regime', 'rd_mom_1', 'ret_1']
corrs = []
for t in TARGETS:
    valid = df.dropna(subset=[t] + [c for c in cols_check if c in df.columns])
    if len(valid) < 100:
        continue
    row = {'target': t}
    for c in cols_check:
        if c in valid.columns:
            row[c] = valid[t].corr(valid[c])
    corrs.append(row)

corr_df = pd.DataFrame(corrs)
print('Корреляция target с rd_regime, rd_mom_1, ret_1 (не должна быть ~1):')
print(corr_df.to_string(index=False))

print('\nTarget vs rd_regime (confusion-like, только BUY/SELL):')
for t in TARGETS:
    v = df.dropna(subset=[t, 'rd_regime'])
    v = v[(v[t] != 0) & (v['rd_regime'].isin([-1, 1]))]
    if len(v) > 0:
        ct = pd.crosstab(v[t], v['rd_regime'], margins=True)
        print(f'\n{t}:')
        print(ct)

Корреляция target с rd_regime, rd_mom_1, ret_1 (не должна быть ~1):


     target  rd_regime  rd_mom_1     ret_1
tb_vol_5bar   0.072870  0.059408 -0.009022
  tp_sl_2_1   0.050185  0.063642  0.001986
 tp_sl_1_05   0.064181  0.057391 -0.012692

Target vs rd_regime (confusion-like, только BUY/SELL):

tb_vol_5bar:
rd_regime        -1       1     All
tb_vol_5bar                        
-1.0          74372   63263  137635
 1.0          63594   75500  139094
 All         137966  138763  276729



tp_sl_2_1:
rd_regime     -1      1     All
tp_sl_2_1                      
-1.0       25807  21553   47360
 1.0       24605  29778   54383
 All       50412  51331  101743

tp_sl_1_05:
rd_regime      -1      1     All
tp_sl_1_05                      
-1.0        52007  43487   95494
 1.0        44905  52969   97874
 All        96912  96456  193368


## 4. Фичи (исследовательские + наши)

- **HSE 2025 (стохастика):** volatility, returns, momentum
- **Стандартные индикаторы:** RSI, MACD, Bollinger (см. docs/08_FEATURES_SOURCES_AND_RATIONALE.md)
- **Собственные:** rd_mom, rd_zscore, volume_rel, OHLC, time

In [5]:
FEATURES = [c for c in get_feature_columns(include_symbol=True) if c in df.columns]
# rd_mom_1 — опционально исключить при риске leakage; для tb_vol обычно ок
print(f'Фичей: {len(FEATURES)}')
print(FEATURES)

Фичей: 25
['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14', 'macd_line', 'macd_signal', 'macd_hist', 'bb_width', 'bb_position', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'symbol_encoded']


## 5. Train/Val/Test по сессиям

In [6]:
sessions_all = list(df['session_key'].unique())
np.random.seed(42)
np.random.shuffle(sessions_all)
n = len(sessions_all)
n_train, n_val = int(0.7 * n), int(0.15 * n)
train_sess = set(sessions_all[:n_train])
val_sess = set(sessions_all[n_train:n_train + n_val])
test_sess = set(sessions_all[n_train + n_val:])
print(f'Sessions: train {len(train_sess)}, val {len(val_sess)}, test {len(test_sess)}')

Sessions: train 733, val 157, test 158


## 6. Математическая (rule-based) модель

Простые правила на основе momentum + RSI + volatility:
- BUY: rd_mom_1 > 0, RSI < 70, volatility выше медианы
- SELL: rd_mom_1 < 0, RSI > 30, volatility выше медианы

In [7]:
def math_model_predict(df_subset, feat_cols):
    """Rule-based: BUY если rd_mom>0, RSI<70, vol>median; SELL симметрично."""
    d = df_subset.copy()
    if 'rd_mom_1' not in d.columns or 'rsi_14' not in d.columns or 'volatility_14' not in d.columns:
        return np.full(len(d), 0.5)
    vol_med = max(d['volatility_14'].median(), 1e-6)
    high_vol = d['volatility_14'].fillna(0) >= vol_med
    rd_pos = d['rd_mom_1'].fillna(0) > 0
    rd_neg = d['rd_mom_1'].fillna(0) < 0
    rsi_ok_buy = (d['rsi_14'].fillna(50) < 70)
    rsi_ok_sell = (d['rsi_14'].fillna(50) > 30)
    buy_signal = rd_pos & rsi_ok_buy & high_vol
    sell_signal = rd_neg & rsi_ok_sell & high_vol
    proba = np.full(len(d), 0.5)
    proba[buy_signal & ~sell_signal] = 0.8
    proba[sell_signal & ~buy_signal] = 0.2
    tie = ~buy_signal & ~sell_signal & high_vol
    proba[tie & rd_pos], proba[tie & rd_neg] = 0.55, 0.45
    return proba

def _run_pnl(proba, bt_df, thresh_hi=0.6, thresh_lo=0.4, commission=0.001):
    """HOLD при proba in [thresh_lo, thresh_hi]. Комиссия только при смене позиции."""
    signal = np.where(proba >= thresh_hi, 1, np.where(proba <= thresh_lo, -1, 0))
    ret = bt_df['ret_next'].values
    pos = np.zeros(len(signal), dtype=int)
    pos[0] = signal[0]
    for i in range(1, len(signal)):
        pos[i] = signal[i] if signal[i] != 0 else pos[i - 1]
    pnl = pos * ret
    changes = np.diff(pos, prepend=pos[0]) != 0
    comm_total = changes.sum() * commission
    return (pnl.sum() - comm_total) * 100, (pos != 0).sum(), changes.sum()

print('math_model_predict, _run_pnl (HOLD + commission on change) defined.')

math_model_predict, _run_pnl (HOLD + commission on change) defined.


## 7. Base модель (CatBoost / LightGBM)

In [8]:
def eval_metrics(name, proba, y_true):
    pred = (proba >= 0.5).astype(int)
    return {
        'model': name,
        'AUC': roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else 0.5,
        'Accuracy': accuracy_score(y_true, pred),
        'F1': f1_score(y_true, pred, zero_division=0),
        'Precision': precision_score(y_true, pred, zero_division=0),
        'Recall': recall_score(y_true, pred, zero_division=0),
    }

all_metrics, all_pnl = [], []
lgb_models, lgb_scalers, cb_models = {}, {}, {}
COMMISSION = 0.001

for TARGET_COL in TARGETS:
    valid = df.dropna(subset=[TARGET_COL]).copy()
    valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
    valid['y'] = (valid[TARGET_COL] == 1).astype(int)
    valid = valid.dropna(subset=FEATURES + ['y'])
    train_df = valid[valid['session_key'].isin(train_sess)].copy()
    val_df = valid[valid['session_key'].isin(val_sess)].copy()
    test_df = valid[valid['session_key'].isin(test_sess)].copy()
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[FEATURES].fillna(0))
    X_test = scaler.transform(test_df[FEATURES].fillna(0))
    y_train, y_test = train_df['y'].values, test_df['y'].values

    bt = test_df.copy()
    bt['ret_next'] = bt.groupby('session_key')['close_price'].pct_change().shift(-1)
    bt = bt.dropna(subset=['ret_next'])
    X_bt = scaler.transform(bt[FEATURES].fillna(0))

    dummy = DummyClassifier(strategy='stratified', random_state=42)
    dummy.fit(X_train, y_train)
    for name, proba in [
        ('Dummy', dummy.predict_proba(X_test)[:, 1]),
        ('Math (rules)', math_model_predict(test_df, FEATURES)),
    ]:
        m = eval_metrics(name, proba, y_test)
        m['target'] = TARGET_COL
        all_metrics.append(m)
    for name, proba in [
        ('Dummy', dummy.predict_proba(X_bt)[:, 1]),
        ('Math (rules)', math_model_predict(bt, FEATURES)),
    ]:
        net, bars_pos, n_changes = _run_pnl(proba, bt, commission=COMMISSION)
        all_pnl.append({'target': TARGET_COL, 'model': name, 'net_%': net, 'trades': n_changes})

    if HAS_LGB:
        lgb_m = lgb.LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=42, verbose=-1)
        lgb_m.fit(X_train, y_train)
        lgb_models[TARGET_COL] = lgb_m
        lgb_scalers[TARGET_COL] = scaler
        lp_test = lgb_m.predict_proba(X_test)[:, 1]
        lp_bt = lgb_m.predict_proba(X_bt)[:, 1]
        all_metrics.append(dict(eval_metrics('LightGBM', lp_test, y_test), target=TARGET_COL))
        n, _, nc = _run_pnl(lp_bt, bt, commission=COMMISSION)
        all_pnl.append({'target': TARGET_COL, 'model': 'LightGBM', 'net_%': n, 'trades': nc})
    if HAS_CB:
        cb_m = cb.CatBoostClassifier(iterations=200, depth=8, learning_rate=0.05, random_state=42, verbose=0)
        cb_m.fit(X_train, y_train)
        cb_models[TARGET_COL] = cb_m
        cp_test = cb_m.predict_proba(X_test)[:, 1]
        cp_bt = cb_m.predict_proba(X_bt)[:, 1]
        all_metrics.append(dict(eval_metrics('CatBoost', cp_test, y_test), target=TARGET_COL))
        n, _, nc = _run_pnl(cp_bt, bt, commission=COMMISSION)
        all_pnl.append({'target': TARGET_COL, 'model': 'CatBoost', 'net_%': n, 'trades': nc})

res_df = pd.DataFrame(all_metrics)
pnl_df = pd.DataFrame(all_pnl)
print('Метрики (test):')
print(res_df.to_string(index=False))

Метрики (test):
       model      AUC  Accuracy       F1  Precision   Recall      target
       Dummy 0.499177  0.499191 0.501916   0.501299 0.502534 tb_vol_5bar
Math (rules) 0.531532  0.520712 0.617429   0.515205 0.770258 tb_vol_5bar
    LightGBM 0.685869  0.630624 0.634990   0.630177 0.639877 tb_vol_5bar
    CatBoost 0.683517  0.629340 0.632581   0.629717 0.635472 tb_vol_5bar
       Dummy 0.508199  0.510303 0.542373   0.529556 0.555826   tp_sl_2_1
Math (rules) 0.530902  0.539520 0.637117   0.541245 0.774263   tp_sl_2_1
    LightGBM 0.692700  0.636697 0.676104   0.632415 0.726277   tp_sl_2_1
    CatBoost 0.685836  0.624347 0.667375   0.620569 0.721817   tp_sl_2_1
       Dummy 0.500180  0.500297 0.508076   0.506072 0.510097  tp_sl_1_05
Math (rules) 0.535230  0.525802 0.620825   0.521282 0.767358  tp_sl_1_05
    LightGBM 0.705329  0.642725 0.653619   0.641393 0.666321  tp_sl_1_05
    CatBoost 0.696713  0.636147 0.649004   0.633817 0.664938  tp_sl_1_05


## 8. Backtest (HOLD 0.4–0.6, комиссия при смене позиции)

In [9]:
# PnL уже посчитан в ячейке 6 (loop). Показать по target:
print('PnL по target (net %, trades = смены позиции):')
for tcol in TARGETS:
    sub = pnl_df[pnl_df['target'] == tcol]
    print(f'\n--- {tcol} ---')
    print(sub[['model', 'net_%', 'trades']].to_string(index=False))

PnL по target (net %, trades = смены позиции):

--- tb_vol_5bar ---
       model        net_%  trades
       Dummy -2114.121666   21009
Math (rules)   -45.956780    6859
    LightGBM  2264.152891    4599
    CatBoost  2143.506064    3695

--- tp_sl_2_1 ---
       model       net_%  trades
       Dummy -629.152899    6997
Math (rules)  161.602587    2298
    LightGBM 1146.476981     955
    CatBoost 1143.173093     864

--- tp_sl_1_05 ---
       model        net_%  trades
       Dummy -1413.407588   14219
Math (rules)   -25.257441    4827
    LightGBM  2011.046260    2768
    CatBoost  1925.650350    2224


## 9. Важность фичей (RD и др.)

Проверяем, как сильно RD-фичи (rd_mom_1, rd_mom_5, rd_mom_10, rd_zscore_30 и т.д.) влияют на предсказание.

In [10]:
RD_COLS = [c for c in FEATURES if c.startswith('rd_') or c == 'abs_rd']
if HAS_LGB and lgb_models:
    imp = pd.DataFrame({
        tcol: dict(zip(FEATURES, lgb_models[tcol].feature_importances_))
        for tcol in TARGETS
    })
    imp['mean'] = imp.mean(axis=1)
    imp = imp.sort_values('mean', ascending=False)
    print('Top-10 фичей по importance (LightGBM):')
    print(imp.head(10).to_string())
    rd_total = imp.loc[imp.index.isin(RD_COLS), 'mean'].sum()
    rd_pct = 100 * rd_total / imp['mean'].sum()
    print(f'\nДоля RD-фичей в общей importance: {rd_pct:.1f}%')
    print(f'RD-фичи: {RD_COLS}')

Top-10 фичей по importance (LightGBM):
                tb_vol_5bar  tp_sl_2_1  tp_sl_1_05        mean
hour_cos                374        705         543  540.666667
bb_position             696        322         538  518.666667
hour_sin                352        635         520  502.333333
rd_mom_5                560        265         383  402.666667
bb_width                394        371         432  399.000000
rd_ema_20               328        426         381  378.333333
symbol_encoded          177        513         378  356.000000
rd_zscore_30            444        275         335  351.333333
rd_mom_10               371        252         336  319.666667
rd_mom_1                348        207         251  268.666667

Доля RD-фичей в общей importance: 33.2%
RD-фичи: ['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd']


## 10. Сводная таблица

In [11]:
merged = res_df.merge(pnl_df, on=['target', 'model'], how='left')
print('=== Итоговое сравнение (все target) ===')
print(merged[['target', 'model', 'AUC', 'F1', 'net_%', 'trades']].to_string(index=False))
for tcol in TARGETS:
    m = merged[merged['target'] == tcol]
    print(f'\n{tcol}: лучшая AUC={m.loc[m["AUC"].idxmax(), "model"]}, лучшая net_%={m.loc[m["net_%"].idxmax(), "model"]}')

=== Итоговое сравнение (все target) ===
     target        model      AUC       F1        net_%  trades
tb_vol_5bar        Dummy 0.499177 0.501916 -2114.121666   21009
tb_vol_5bar Math (rules) 0.531532 0.617429   -45.956780    6859
tb_vol_5bar     LightGBM 0.685869 0.634990  2264.152891    4599
tb_vol_5bar     CatBoost 0.683517 0.632581  2143.506064    3695
  tp_sl_2_1        Dummy 0.508199 0.542373  -629.152899    6997
  tp_sl_2_1 Math (rules) 0.530902 0.637117   161.602587    2298
  tp_sl_2_1     LightGBM 0.692700 0.676104  1146.476981     955
  tp_sl_2_1     CatBoost 0.685836 0.667375  1143.173093     864
 tp_sl_1_05        Dummy 0.500180 0.508076 -1413.407588   14219
 tp_sl_1_05 Math (rules) 0.535230 0.620825   -25.257441    4827
 tp_sl_1_05     LightGBM 0.705329 0.653619  2011.046260    2768
 tp_sl_1_05     CatBoost 0.696713 0.649004  1925.650350    2224

tb_vol_5bar: лучшая AUC=LightGBM, лучшая net_%=LightGBM

tp_sl_2_1: лучшая AUC=LightGBM, лучшая net_%=LightGBM

tp_sl_1_05: луч

## 11. Анализ неоднозначных баров (intrabar TP+SL)

Когда в **одном** баре и high касается TP, и low касается SL — порядок событий неизвестен. Оцениваем масштаб: сколько баров и entry-points затронуто.

In [12]:
def count_ambiguous_bars(df, TP_pct, SL_pct, H=20, by='session_key'):
    """Считает ambiguous бары и entry-points, где first-hit бар ambiguous."""
    stats = {
        'n_entries': 0,
        'n_bars_total': 0,
        'n_ambig_long': 0,
        'n_ambig_short': 0,
        'n_entries_ambig_first_long': 0,
        'n_entries_ambig_first_short': 0,
    }
    for skey, g in df.groupby(by, sort=False):
        c = g['close_price'].values
        h_arr = g['high'].values
        l_arr = g['low'].values
        for i in range(len(g) - H):
            entry = c[i]
            tp_up, sl_down = entry * (1 + TP_pct), entry * (1 - SL_pct)
            tp_dn, sl_up = entry * (1 - TP_pct), entry * (1 + SL_pct)
            stats['n_entries'] += 1
            stats['n_bars_total'] += H
            first_long_j, first_short_j = None, None
            for j in range(1, H + 1):
                hi, lo = h_arr[i + j], l_arr[i + j]
                ambig_long = (hi >= tp_up) and (lo <= sl_down)
                ambig_short = (hi >= sl_up) and (lo <= tp_dn)
                if ambig_long:
                    stats['n_ambig_long'] += 1
                if ambig_short:
                    stats['n_ambig_short'] += 1

                if first_long_j is None and ((hi >= tp_up) or (lo <= sl_down)):
                    first_long_j = j
                    if ambig_long:
                        stats['n_entries_ambig_first_long'] += 1
                if first_short_j is None and ((lo <= tp_dn) or (hi >= sl_up)):
                    first_short_j = j
                    if ambig_short:
                        stats['n_entries_ambig_first_short'] += 1
    return stats

configs = [
    ('tp_sl_2_1', 0.02, 0.01),
    ('tp_sl_1_05', 0.01, 0.005),
]
print('=== Неоднозначные бары (high касается TP и low касается SL в одном баре) ===\n')
for name, tp, sl in configs:
    s = count_ambiguous_bars(df, tp, sl)
    pct_bars = 100 * (s['n_ambig_long'] + s['n_ambig_short']) / max(1, s['n_bars_total'])
    pct_entries_long = 100 * s['n_entries_ambig_first_long'] / max(1, s['n_entries'])
    pct_entries_short = 100 * s['n_entries_ambig_first_short'] / max(1, s['n_entries'])
    print(f'--- {name} (TP={tp*100:.1f}%, SL={sl*100:.1f}%) ---')
    print(f'  Entry-points: {s["n_entries"]:,}, баров в horizon: {s["n_bars_total"]:,}')
    print(f'  Ambiguous баров: long {s["n_ambig_long"]:,}, short {s["n_ambig_short"]:,} ({pct_bars:.2f}% всех)')
    print(f'  Entry где первый сработавший бар — ambiguous: long {s["n_entries_ambig_first_long"]:,} ({pct_entries_long:.1f}%), short {s["n_entries_ambig_first_short"]:,} ({pct_entries_short:.1f}%)')
    print()

=== Неоднозначные бары (high касается TP и low касается SL в одном баре) ===



--- tp_sl_2_1 (TP=2.0%, SL=1.0%) ---
  Entry-points: 344,415, баров в horizon: 6,888,300
  Ambiguous баров: long 2,628, short 2,575 (0.08% всех)
  Entry где первый сработавший бар — ambiguous: long 427 (0.1%), short 379 (0.1%)



--- tp_sl_1_05 (TP=1.0%, SL=0.5%) ---
  Entry-points: 344,415, баров в horizon: 6,888,300
  Ambiguous баров: long 13,547, short 13,237 (0.39% всех)
  Entry где первый сработавший бар — ambiguous: long 1,900 (0.6%), short 1,852 (0.5%)



## 12. Выводы и рекомендации

### Сравнение target

| Target | net_% (LightGBM) | trades | AUC |
|--------|------------------|--------|-----|
| **tb_vol_5bar** | +2264% | 4599 | 0.69 |
| **tp_sl_2_1** (TP 2%, SL 1%) | +1146% | 955  | 0.69 |
| **tp_sl_1_05** (TP 1%, SL 0.5%, nikitapre) | +2011% | 2768 | 0.71 |

**tb_vol_5bar:** максимум прибыли, но 4.8× больше сделок, чем tp_sl_2_1. Выше turnover и риск шума.

**tp_sl_2_1:** меньше сделок (955), умеренная прибыль. Строгие барьеры 2%/1%.

**tp_sl_1_05 (nikitapre):** выше AUC (0.71) и net_% (+2011%), чем tp_sl_2_1. Больше сделок (2768), чем у 2%/1%, но меньше, чем у tb_vol. Тighter барьеры 1%/0.5% дают больше сигналов при сохранении качества.

### Роль RD-фичей

RD-фичи (rd_mom_1, rd_mom_5, rd_mom_10, rd_zscore_30 и др.) участвуют в предсказании (~33% importance). Math-модель опирается на rd_mom_1; LightGBM/CatBoost используют полный набор RD. Math даёт +162% на tp_sl_2_1; ML-модели усиливают эффект.

### Рекомендация для продакшена

**Рекомендуется: target tp_sl_1_05, модель LightGBM (или CatBoost).**

Причины:
1. **Лучший AUC (0.71)** среди всех target.
2. **Высокий net_% (+2011%)** при 2768 сменах позиции.
3. **Меньше сделок**, чем tb_vol — ниже издержки и риск проскальзывания.
4. **1%/0.5%** — проверенный RR из nikitapre, precision ≥ 40% при c=0.1%.

tb_vol_5bar даёт чуть больше прибыли (+2264% vs +2011%), но при 1.7× больше сделок — выше риск проскальзывания и переобучения.

**Следующие шаги:** paper trading на tp_sl_1_05 + LightGBM, мониторинг precision, Sharpe, drawdown. 
ambiguous intrabar учтены как HOLD (0), влияние на итоговые метрики ограниченное (порядка <1% entry-points)

## 13. Сохранение финального выбора (tp_sl_1_05 + LightGBM)

Сохраняем датасет с выбранной целевой переменной и baseline-артефакты для последующего использования в пайплайне.

In [13]:
SELECTED_TARGET = 'tp_sl_1_05'
SELECTED_MODEL = 'LightGBM'

out_dir = os.path.join(_root, 'outputs')
os.makedirs(out_dir, exist_ok=True)

# 1) Финальный parquet с выбранной target (и унифицированной колонкой `target`).
final_df = df.dropna(subset=[SELECTED_TARGET]).copy()
final_df['target'] = final_df[SELECTED_TARGET].astype(np.float64)
final_parquet = os.path.join(out_dir, f'prepared_with_target_{SELECTED_TARGET}.parquet')
final_df.to_parquet(final_parquet, index=False, compression='snappy')

# 2) Сводка выбора target/model.
selection_summary = merged[(merged['target'] == SELECTED_TARGET) & (merged['model'] == SELECTED_MODEL)].copy()
summary_csv = os.path.join(out_dir, 'target_selection_summary.csv')
selection_summary.to_csv(summary_csv, index=False)

# 3) Baseline LightGBM + scaler для выбранного target.
if HAS_LGB and SELECTED_TARGET in lgb_models:
    model_bundle = {
        'model_name': 'LightGBM',
        'target': SELECTED_TARGET,
        'features': FEATURES,
        'scaler': lgb_scalers[SELECTED_TARGET],
        'model': lgb_models[SELECTED_TARGET],
    }
    model_path = os.path.join(out_dir, f'baseline_lgbm_{SELECTED_TARGET}.joblib')
    joblib.dump(model_bundle, model_path)
    print(f'Сохранена baseline-модель: {model_path}')
else:
    model_path = None
    print('LightGBM недоступен — baseline-модель не сохранена.')

print(f'Сохранён датасет: {final_parquet}')
print(f'Сохранена сводка: {summary_csv}')
print(f'Строк в финальном датасете: {len(final_df):,}')

Сохранена baseline-модель: c:\project\trading_bot_2Engine\outputs\baseline_lgbm_tp_sl_1_05.joblib
Сохранён датасет: c:\project\trading_bot_2Engine\outputs\prepared_with_target_tp_sl_1_05.parquet
Сохранена сводка: c:\project\trading_bot_2Engine\outputs\target_selection_summary.csv
Строк в финальном датасете: 344,415
